In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from constants import FILE

In [2]:
path = Path()
file_path = path.absolute() / 'out' / FILE

In [3]:
EXCLUDED_WEIGHTS = [
    '[1, 0, 0, 0, 0]',
    '[0, 1, 0, 0, 0]',
    '[0, 0, 1, 0, 0]',
    '[0, 0, 0, 1, 0]',
    '[0, 0, 0, 0, 1]',
]

In [4]:
df = pd.read_excel(file_path, engine='openpyxl')

In [5]:
df = df[~df["weight"].isin(EXCLUDED_WEIGHTS)].copy()

In [6]:
df = df[~df['instancia'].isna()]

In [7]:
hash_list = df["weight_hash"].unique()

def make_weight_hash_map_from_list(hash_list, start_at=1):
    return {h: rf"$w_{{{i}}}$" for i, h in enumerate(hash_list, start=start_at)}

hash_map = make_weight_hash_map_from_list(hash_list)

In [8]:
df["weight_label"] = df["weight_hash"].map(hash_map).fillna(df["weight_hash"])

In [9]:
df["gap"] = df["gap"].clip(lower=0)

In [10]:
# =========================
# 1) Preparação dos dados
# =========================
# Troque "df" pelo nome do seu DataFrame, se necessário
df["gap"] = df["gap"].clip(lower=0)
base = df.copy()

cols = ["instancia", "clientes", "classe", "gap"]
faltantes = [c for c in cols if c not in base.columns]
if faltantes:
    raise ValueError(f"Colunas ausentes no DataFrame: {faltantes}")

dados = base[cols].copy()
dados["clientes"] = pd.to_numeric(dados["clientes"], errors="coerce")
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")
dados = dados.dropna(subset=["clientes", "classe", "gap"])

# Ajustes de tipo
dados["clientes"] = dados["clientes"].astype(int)
dados["classe"] = dados["classe"].astype(str)


dados["gap_pct"] = dados["gap"] * 100

# =========================
# 2) Resumo estatístico
# =========================
resumo = (
    dados.groupby(["clientes", "classe"], as_index=False)
    .agg(
        n_execucoes=("gap_pct", "size"),
        n_instancias=("instancia", "nunique"),
        gap_medio_pct=("gap_pct", "mean"),
        gap_mediano_pct=("gap_pct", "median"),
        gap_std_pct=("gap_pct", "std"),
        gap_p90_pct=("gap_pct", lambda s: s.quantile(0.90)),
        gap_min_pct=("gap_pct", "min"),
        gap_max_pct=("gap_pct", "max"),
    )
    .sort_values(["clientes", "classe"])
)

display(resumo.round(3))

# Tabelas dinâmicas
tabela_media = dados.pivot_table(
    index="clientes", columns="classe", values="gap_pct", aggfunc="mean"
).sort_index()

tabela_qtd = dados.pivot_table(
    index="clientes", columns="classe", values="gap_pct", aggfunc="size", fill_value=0
).sort_index()

display(tabela_media.round(3))
display(tabela_qtd)

# =========================
# 3) Plotly - Boxplot
# =========================
fig_box = px.box(
    dados.sort_values("clientes"),
    x="clientes",
    y="gap_pct",
    color="classe",
    points="outliers",
    labels={
        "clientes": "Quantidade de clientes",
        "gap_pct": "GAP (%)",
        "classe": "Classe",
    },
    title="Distribuição do GAP (%) por quantidade de clientes e classe",
)

fig_box.update_layout(
    boxmode="group",
    legend_title_text="Classe",
    template="plotly_white"
)
fig_box.show()

# =========================
# 4) Plotly - Heatmap (média)
# =========================
fig_heat = px.imshow(
    tabela_media,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="YlOrRd",
    labels=dict(x="Classe", y="Quantidade de clientes", color="GAP médio (%)"),
    title="Heatmap da média do GAP (%) - clientes x classe",
)

fig_heat.update_layout(template="plotly_white")
fig_heat.show()

,clientes,classe,n_execucoes,n_instancias,gap_medio_pct,gap_mediano_pct,gap_std_pct,gap_p90_pct,gap_min_pct,gap_max_pct
0,5,1,3636,10,2.601,0.036,5.146,8.913,0.000,47.513
1,5,2,3600,10,2.132,0.026,4.475,7.034,0.000,40.820
2,5,3,3600,10,3.304,0.051,6.075,10.013,0.000,49.782
3,10,1,3600,10,58.424,63.175,21.907,80.790,0.049,89.810
4,10,2,3600,10,56.408,61.678,22.420,79.467,0.034,95.503
5,10,3,3600,10,57.276,62.786,21.153,79.891,0.000,92.707
6,20,1,6756,10,93.706,94.311,3.306,97.263,81.234,99.058
7,20,2,6588,10,94.353,94.879,3.046,97.748,79.525,99.717
8,20,3,6756,10,93.949,94.605,3.534,97.710,71.172,99.096
9,30,1,24,2,96.720,96.720,1.171,97.866,95.574,97.866


classe,1,2,3
clientes,,,
5,2.601,2.132,3.304
10,58.424,56.408,57.276
20,93.706,94.353,93.949
30,96.720,97.867,95.914


classe,1,2,3
clientes,,,
5,3636,3600,3600
10,3600,3600,3600
20,6756,6588,6756
30,24,36,108


In [11]:
# =========================
# 1) Preparação dos dados
# =========================
# Troque "df" pelo nome do seu DataFrame, se necessário
base = df.copy()

required_cols = ["instancia", "alpha", "clientes", "classe", "weight_label", "gap"]
missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

dados = base[required_cols].copy()

dados["alpha"] = pd.to_numeric(dados["alpha"], errors="coerce")
dados["clientes"] = pd.to_numeric(dados["clientes"], errors="coerce")
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")

dados = dados.dropna(subset=["alpha", "clientes", "classe", "weight_label", "gap"])

dados["clientes"] = dados["clientes"].astype(int)
dados["classe"] = dados["classe"].astype(str)
dados["weight_label"] = dados["weight_label"].astype(str)

# Regra pedida: gap negativo vira 0
dados["gap"] = dados["gap"].clip(lower=0)

# Converte para percentual se estiver em fração
if dados["gap"].abs().max() <= 1.5:
    dados["gap_pct"] = dados["gap"] * 100
else:
    dados["gap_pct"] = dados["gap"]

# =========================
# 2) Resumo da análise
# =========================
resumo = (
    dados.groupby(["alpha", "clientes", "classe", "weight_label"], as_index=False)
    .agg(
        n_execucoes=("gap_pct", "size"),
        n_instancias=("instancia", "nunique"),
        gap_medio_pct=("gap_pct", "mean"),
        gap_mediano_pct=("gap_pct", "median"),
        gap_std_pct=("gap_pct", "std"),
        gap_p90_pct=("gap_pct", lambda s: s.quantile(0.90)),
        gap_min_pct=("gap_pct", "min"),
        gap_max_pct=("gap_pct", "max"),
    )
    .sort_values(["alpha", "clientes", "classe", "weight_label"])
)

display(resumo.round(3))

# Tabela dinâmica útil para inspeção
tabela_media = resumo.pivot_table(
    index=["alpha", "clientes"],
    columns=["classe", "weight_label"],
    values="gap_medio_pct"
).sort_index()

display(tabela_media.round(3))

# =========================
# 3) Plotly - visão geral com alpha no slider
# =========================
resumo["alpha_frame"] = resumo["alpha"].map(lambda x: f"{x:g}")

fig_line = px.line(
    resumo,
    x="clientes",
    y="gap_medio_pct",
    color="classe",
    line_dash="weight_label",
    markers=True,
    animation_frame="alpha_frame",
    hover_data={
        "n_execucoes": True,
        "n_instancias": True,
        "gap_mediano_pct": ":.2f",
        "gap_p90_pct": ":.2f",
        "gap_std_pct": ":.2f",
    },
    labels={
        "clientes": "Quantidade de clientes",
        "gap_medio_pct": "GAP médio (%)",
        "classe": "Classe",
        "weight_label": "Weight label",
        "alpha_frame": "Alpha",
    },
    title="GAP médio (%) por clientes, classe e weight_label (slider por alpha)",
)

fig_line.update_layout(template="plotly_white", legend_title_text="Classe")
fig_line.show()

# =========================
# 4) Função de heatmap (alpha + weight_label específicos)
# =========================
def plot_heatmap(alpha_sel, weight_sel):
    corte = resumo[
        (resumo["alpha"] == alpha_sel) &
        (resumo["weight_label"] == str(weight_sel))
    ]

    if corte.empty:
        print(f"Sem dados para alpha={alpha_sel} e weight_label={weight_sel}")
        return

    pivot = (
        corte.pivot(index="clientes", columns="classe", values="gap_medio_pct")
        .sort_index()
    )

    fig_heat = px.imshow(
        pivot,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="YlOrRd",
        labels={
            "x": "Classe",
            "y": "Quantidade de clientes",
            "color": "GAP médio (%)",
        },
        title=f"Heatmap do GAP médio (%) | alpha={alpha_sel} | weight_label={weight_sel}",
    )
    fig_heat.update_layout(template="plotly_white")
    fig_heat.show()

# Exemplo de uso:
plot_heatmap(alpha_sel=resumo["alpha"].iloc[0], weight_sel=resumo["weight_label"].iloc[0])

,alpha,clientes,classe,weight_label,n_execucoes,n_instancias,gap_medio_pct,gap_mediano_pct,gap_std_pct,gap_p90_pct,gap_min_pct,gap_max_pct
0,0.01,5,1,$w_{1}$,303,10,4.050,2.892,4.426,10.412,0.000,17.325
1,0.01,5,1,$w_{2}$,303,10,3.740,2.102,4.177,9.764,0.003,13.669
2,0.01,5,1,$w_{3}$,303,10,3.607,2.655,3.810,8.346,0.000,14.949
3,0.01,5,1,$w_{4}$,303,10,6.987,7.233,6.890,14.298,0.006,36.366
4,0.01,5,1,$w_{5}$,303,10,3.845,1.547,4.938,10.895,0.004,21.835
...,...,...,...,...,...,...,...,...,...,...,...,...
103,0.99,20,3,$w_{2}$,456,10,91.706,93.779,5.150,96.092,71.172,96.726
104,0.99,20,3,$w_{3}$,504,10,94.950,95.672,2.278,97.151,88.198,97.895
105,0.99,20,3,$w_{4}$,600,10,96.539,97.196,1.973,98.225,88.437,98.920
106,0.99,20,3,$w_{5}$,516,10,96.549,97.181,1.753,98.065,90.133,98.560


classe               1                                               2  \
weight_label   $w_{1}$ $w_{2}$ $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$ $w_{1}$   
alpha clientes                                                           
0.01  5          4.050   3.740   3.607   6.987   3.845   6.214   4.296   
      10        61.958  54.587  53.557  66.550  61.801  68.263  63.082   
      20        92.513  89.168  89.935  92.748  93.719  94.232  93.282   
0.99  5          0.272   0.275   0.024   1.207   0.031   0.962   0.108   
      10        60.597  41.296  55.744  68.142  40.826  67.773  56.790   
      20        95.160  93.290  94.666  96.308  96.266  97.348  95.519   

classe                                                       3          \
weight_label   $w_{2}$ $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$ $w_{1}$ $w_{2}$   
alpha clientes                                                           
0.01  5          2.732   2.937   4.873   3.299   4.821   4.813   5.612   
      10        51.779  55.267  65.755  60.362  68.826  62.818  53.237   
      20        90.477  91.001  93.639  94.445  94.715  92.730  89.557   
0.99  5          0.186   0.136   1.708   0.063   0.425   1.227   0.393   
      10        41.391  42.646  67.361  38.206  65.434  60.287  43.022   
      20        93.916  95.358  96.565  96.508  97.680  95.589  91.706   

classe                                          
weight_label   $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$  
alpha clientes                                  
0.01  5          4.302   8.248   3.615   7.329  
      10        51.508  62.643  60.460  66.775  
      20        90.779  93.273  93.900  94.583  
0.99  5          0.115   2.709   0.870   0.410  
      10        47.122  69.204  40.249  69.984  
      20        94.950  96.539  96.549  97.507

In [12]:
# Assume que df, pd e px já estão disponíveis no notebook

base = df.copy()

# Usa weight_label; fallback para weigh_label se vier com esse nome
peso_col = "weight_label" if "weight_label" in base.columns else "weigh_label"
if peso_col not in base.columns:
    raise ValueError("Coluna de peso não encontrada: 'weight_label' (ou 'weigh_label').")

dados = base[[peso_col, "gap"]].copy()
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")
dados = dados.dropna(subset=[peso_col, "gap"])

# Gap negativo vira zero
dados["gap"] = dados["gap"].clip(lower=0)

# Converte para percentual se necessário
if dados["gap"].abs().max() <= 1.5:
    dados["gap_pct"] = dados["gap"] * 100
else:
    dados["gap_pct"] = dados["gap"]

dados[peso_col] = dados[peso_col].astype(str)

# Remove extremos por grupo (IQR por weight_label)
iqr_stats = (
    dados.groupby(peso_col)["gap_pct"]
    .quantile([0.25, 0.75]).unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)
iqr_stats["iqr"] = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lim_inf"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["lim_sup"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

dados_aux = dados.join(iqr_stats[["lim_inf", "lim_sup"]], on=peso_col)
dados_sem_extremos = dados_aux[
    (dados_aux["gap_pct"] >= dados_aux["lim_inf"]) &
    (dados_aux["gap_pct"] <= dados_aux["lim_sup"])
].drop(columns=["lim_inf", "lim_sup"])

# Resumo
resumo = (
    dados_sem_extremos.groupby(peso_col, as_index=False)
    .agg(
        n_execucoes=("gap_pct", "size"),
        gap_medio_pct=("gap_pct", "mean"),
        gap_mediano_pct=("gap_pct", "median"),
        gap_std_pct=("gap_pct", "std"),
        gap_p90_pct=("gap_pct", lambda s: s.quantile(0.90)),
        gap_min_pct=("gap_pct", "min"),
        gap_max_pct=("gap_pct", "max"),
    )
    .sort_values(peso_col)
)

display(resumo.round(3))

# Boxplot sem extremos
fig_box = px.box(
    dados_sem_extremos,
    x=peso_col,
    y="gap_pct",
    points=False,
    labels={peso_col: "weight_label", "gap_pct": "GAP (%)"},
    title="Distribuição do GAP (%) por weight_label (sem valores extremos - IQR)",
)
fig_box.update_layout(template="plotly_white")
fig_box.show()

,weight_label,n_execucoes,gap_medio_pct,gap_mediano_pct,gap_std_pct,gap_p90_pct,gap_min_pct,gap_max_pct
0,$w_{1}$,7074,62.280,83.410,38.641,96.388,0.0,99.470
1,$w_{2}$,6834,56.167,65.634,38.353,94.626,0.0,99.010
2,$w_{3}$,6846,57.685,75.730,39.296,95.503,0.0,99.152
3,$w_{4}$,7170,64.962,87.272,38.024,97.289,0.0,99.576
4,$w_{5}$,6870,58.890,76.458,41.222,97.096,0.0,99.435
5,$w_{6}$,7110,65.337,89.439,39.400,97.960,0.0,99.717


In [13]:
dados_aux

,weight_label,gap,gap_pct,lim_inf,lim_sup
0,$w_{1}$,0.955737,95.573736,-115.246158,219.995451
1,$w_{1}$,0.955737,95.573736,-115.246158,219.995451
2,$w_{1}$,0.955737,95.573736,-115.246158,219.995451
3,$w_{1}$,0.955737,95.573736,-115.246158,219.995451
4,$w_{1}$,0.955737,95.573736,-115.246158,219.995451
...,...,...,...,...,...
41899,$w_{1}$,0.556454,55.645382,-115.246158,219.995451
41900,$w_{1}$,0.556454,55.645382,-115.246158,219.995451
41901,$w_{1}$,0.556454,55.645382,-115.246158,219.995451
41902,$w_{1}$,0.556454,55.645382,-115.246158,219.995451


In [14]:
dados = base[["weight_label", "gap"]].copy()
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")
dados["gap_pct"] = dados["gap"] * 100
dados = dados.dropna(subset=["weight_label", "gap"])

iqr_stats = (
    dados.groupby("weight_label")["gap_pct"]
    .quantile([0.25, 0.75]).unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)
iqr_stats["iqr"] = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lim_inf"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["lim_sup"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

dados_aux = dados.join(iqr_stats[["lim_inf", "lim_sup"]], on="weight_label")


# Marca outliers pelo critério IQR
dados_aux["is_outlier"] = (
    (dados_aux["gap_pct"] < dados_aux["lim_inf"]) |
    (dados_aux["gap_pct"] > dados_aux["lim_sup"])
)

# ---- Resumo geral (absoluto e %) ----
total_geral = len(dados_aux)
outliers_geral = int(dados_aux["is_outlier"].sum())
pct_outliers_geral = (100 * outliers_geral / total_geral) if total_geral else 0.0

print(f"Outliers (geral): {outliers_geral} de {total_geral} ({pct_outliers_geral:.2f}%)")

# ---- Resumo por weight_label (absoluto e %) ----
resumo_outliers = (
    dados_aux.groupby("weight_label", as_index=False)
    .agg(
        total_registros=("is_outlier", "size"),
        qtd_outliers=("is_outlier", "sum"),
    )
)

resumo_outliers["pct_outliers"] = (
    100 * resumo_outliers["qtd_outliers"] / resumo_outliers["total_registros"]
)

display(
    resumo_outliers.sort_values("qtd_outliers", ascending=False).round(2)
)

# Mantém somente não-outliers
dados_sem_extremos = dados_aux[
    ~dados_aux["is_outlier"]
].drop(columns=["lim_inf", "lim_sup", "is_outlier"])

Outliers (geral): 0 de 41904 (0.00%)


,weight_label,total_registros,qtd_outliers,pct_outliers
0,$w_{1}$,7074,0,0.0
1,$w_{2}$,6834,0,0.0
2,$w_{3}$,6846,0,0.0
3,$w_{4}$,7170,0,0.0
4,$w_{5}$,6870,0,0.0
5,$w_{6}$,7110,0,0.0


In [15]:
df.columns

Index(['time', 'file', 'hash_file', 'weight_hash', 'weight', 'FO', 'gap',
       'solver_time', 'f1', 'f2', 'f3', 'f4', 'f5', 'p1', 'p2', 'p3', 'p4',
       'p5', 'epsilon', 'total_production', 'total_inventory', 'total_setup',
       'total_delivered', 'csetup', 'cprod', 'instancia', 'clientes',
       'produtos', 'veiculos', 'periodos', 'seeds', 'classe', 'new_f1_target',
       'new_f2_target', 'new_f3_target', 'new_f4_target', 'new_f5_target',
       'hash_row', '__source_file__', 'alpha', 'f1_target', 'f2_target',
       'f3_target', 'f4_target', 'f5_target', 'weight_label'],
      dtype='object')

In [16]:
# Assumindo `df` e `pd` já existem no notebook

# =========================================
# 1) Preparação + gap_pct + remoção outliers
# =========================================
base = df.copy()

# Corrige possível typo de schema
if "weight_label" not in base.columns and "weigh_label" in base.columns:
    base = base.rename(columns={"weigh_label": "weight_label"})
elif "weight_label" in base.columns and "weigh_label" in base.columns:
    base["weight_label"] = base["weight_label"].fillna(base["weigh_label"])
    base = base.drop(columns=["weigh_label"])

required_cols = ["instancia", "alpha", "clientes", "classe", "weight_label", "gap"]
missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

dados = base[required_cols].copy()

dados["alpha"] = pd.to_numeric(dados["alpha"], errors="coerce")
dados["clientes"] = pd.to_numeric(dados["clientes"], errors="coerce")
dados["classe"] = pd.to_numeric(dados["classe"], errors="coerce")
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")

dados = dados.dropna(subset=["alpha", "clientes", "classe", "weight_label", "gap"])
dados["clientes"] = dados["clientes"].astype(int)
dados["classe"] = dados["classe"].astype(int).astype(str)
dados["weight_label"] = dados["weight_label"].astype(str)

# Regra: gap negativo vira zero
dados["gap"] = dados["gap"].clip(lower=0)

# Gap em percentual (sempre gap * 100)
dados["gap_pct"] = dados["gap"] * 100

# Remove extremos por grupo (IQR por weight_label)
iqr_stats = (
    dados.groupby("weight_label")["gap_pct"]
    .quantile([0.25, 0.75]).unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)
iqr_stats["iqr"] = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lim_inf"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["lim_sup"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

dados_aux = dados.join(iqr_stats[["lim_inf", "lim_sup"]], on="weight_label")
dados_sem_extremos = dados_aux[
    (dados_aux["gap_pct"] >= dados_aux["lim_inf"]) &
    (dados_aux["gap_pct"] <= dados_aux["lim_sup"])
].drop(columns=["lim_inf", "lim_sup"])

# =========================================
# 2) Resumo + linha All instances
# =========================================
resumo_cls = (
    dados_sem_extremos
    .groupby(["alpha", "clientes", "classe", "weight_label"], as_index=False)
    .agg(gap_medio_pct=("gap_pct", "mean"))
)

resumo_all = (
    dados_sem_extremos
    .groupby(["alpha", "clientes", "weight_label"], as_index=False)
    .agg(gap_medio_pct=("gap_pct", "mean"))
)
resumo_all["classe"] = "all"

resumo = pd.concat([resumo_cls, resumo_all], ignore_index=True)

# Linhas: clientes, classe | Colunas: alpha, weight_label
tabela_media = resumo.pivot_table(
    index=["clientes", "classe"],
    columns=["alpha", "weight_label"],
    values="gap_medio_pct",
    aggfunc="mean"
)

display(tabela_media.round(3))

# =========================================
# 3) Função para gerar LaTeX
# =========================================
def gap_table_to_latex_alpha_weight(
    tabela_media: pd.DataFrame,
    clientes_order=None,
    classes_order=None,
    alpha_order=None,
    weight_order=None,
    class_roman_map=None,
    caption=None,
    label=None,
    table_env=True,
):
    tab = tabela_media.copy()

    if not isinstance(tab.index, pd.MultiIndex) or tab.index.nlevels != 2:
        raise ValueError("tabela_media precisa ter índice MultiIndex: (clientes, classe).")
    if not isinstance(tab.columns, pd.MultiIndex) or tab.columns.nlevels != 2:
        raise ValueError("tabela_media precisa ter colunas MultiIndex: (alpha, weight_label).")

    tab.index = tab.index.set_names(["clientes", "classe"])
    tab.columns = tab.columns.set_names(["alpha", "weight_label"])

    # Helpers definidos antes do uso (corrige UnboundLocalError)
    def num_key(x):
        try:
            return (0, float(x))
        except Exception:
            return (1, str(x))

    def weight_key(w):
        s = str(w)
        digits = "".join(ch for ch in s if ch.isdigit())
        return (0, int(digits)) if digits else (1, s)

    if clientes_order is None:
        clientes_order = sorted(tab.index.get_level_values("clientes").unique().tolist(), key=num_key)

    raw_classes = tab.index.get_level_values("classe").unique().tolist()
    has_all = any(str(c).lower() == "all" for c in raw_classes)

    if classes_order is None:
        sem_all = [c for c in raw_classes if str(c).lower() != "all"]
        classes_order = sorted(sem_all, key=num_key)
        if has_all:
            classes_order.append("all")
    else:
        classes_order = list(classes_order)
        if has_all and not any(str(c).lower() == "all" for c in classes_order):
            classes_order.append("all")

    if alpha_order is None:
        alpha_order = sorted(tab.columns.get_level_values("alpha").unique().tolist(), key=num_key)
    if weight_order is None:
        weight_order = sorted(tab.columns.get_level_values("weight_label").unique().tolist(), key=weight_key)

    if class_roman_map is None:
        class_roman_map = {"1": "I", "2": "II", "3": "III", "4": "IV", 1: "I", 2: "II", 3: "III", 4: "IV"}

    full_rows = pd.MultiIndex.from_product(
        [clientes_order, classes_order], names=["clientes", "classe"]
    )
    full_cols = pd.MultiIndex.from_product(
        [alpha_order, weight_order], names=["alpha", "weight_label"]
    )
    mat = tab.reindex(index=full_rows, columns=full_cols)

    def alpha_label(a):
        try:
            return rf"$\alpha={float(a):g}$"
        except Exception:
            return rf"$\alpha={a}$"

    def class_label(cl):
        if str(cl).lower() == "all":
            return "All instances"
        return class_roman_map.get(cl, class_roman_map.get(str(cl), str(cl)))

    def clients_label(c):
        try:
            return f"{int(float(c))} clients"
        except Exception:
            return f"{c} clients"

    def fmt_pct(v):
        if pd.isna(v):
            return "-"
        return f"{v:.1f}".replace(".", ",") + r"\%"

    A = len(alpha_order)
    W = len(weight_order)
    colspec = "ll" + "c" * (A * W)

    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")

    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")

    # Header 1: blocos por alpha
    row1 = [r"\multicolumn{1}{c}{\textbf{}}", r"\multicolumn{1}{c}{\textbf{}}"]
    row1 += [rf"\multicolumn{{{W}}}{{c}}{{\textbf{{{alpha_label(a)}}}}}" for a in alpha_order]
    lines.append(" & ".join(row1) + r" \\")

    # cmidrule por alpha
    cmid = []
    start = 3
    for _ in alpha_order:
        end = start + W - 1
        cmid.append(rf"\cmidrule(lr){{{start}-{end}}}")
        start = end + 1
    lines.append(" ".join(cmid))

    # Header 2: pesos
    row2 = [r"\multicolumn{1}{c}{\textbf{Clientes}}", r"\multicolumn{1}{c}{\textbf{Classe}}"]
    for _a in alpha_order:
        for w in weight_order:
            row2.append(rf"\multicolumn{{1}}{{c}}{{\textbf{{{w}}}}}")
    lines.append(" & ".join(row2) + r" \\")
    lines.append(r"\midrule")

    # Corpo: linhas por clientes e classe (incluindo all)
    for i, c in enumerate(clientes_order):
        rows_c = [(c, cl) for cl in classes_order]
        nrows = len(rows_c)

        for j, (_, cl) in enumerate(rows_c):
            row = [rf"\multirow{{{nrows}}}{{*}}{{\textbf{{{clients_label(c)}}}}}"] if j == 0 else [""]
            row.append(class_label(cl))

            for a in alpha_order:
                for w in weight_order:
                    row.append(fmt_pct(mat.loc[(c, cl), (a, w)]))

            lines.append(" & ".join(row) + r" \\")

        if i != len(clientes_order) - 1:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if table_env:
        lines.append(r"\end{table}")

    return "\n".join(lines)

# =========================================
# 4) Gerar LaTeX
# =========================================
latex_code = gap_table_to_latex_alpha_weight(
    tabela_media,
    caption="Average gap (\\%) by clients/class (rows) and $\\alpha$/weight (columns), without outliers.",
    label="tab:gap_alpha_weight",
    table_env=True,
)

print(latex_code)

alpha              0.01                                            0.99  \
weight_label    $w_{1}$ $w_{2}$ $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$ $w_{1}$   
clientes classe                                                           
5        1        4.050   3.740   3.607   6.987   3.845   6.214   0.272   
         2        4.296   2.732   2.937   4.873   3.299   4.821   0.108   
         3        4.813   5.612   4.302   8.248   3.615   7.329   1.227   
         all      4.385   4.027   3.615   6.704   3.587   6.122   0.535   
10       1       61.958  54.587  53.557  66.550  61.801  68.263  60.597   
         2       63.082  51.779  55.267  65.755  60.362  68.826  56.790   
         3       62.818  53.237  51.508  62.643  60.460  66.775  60.287   
         all     62.619  53.201  53.444  64.982  60.874  67.954  59.225   
20       1       92.513  89.168  89.935  92.748  93.719  94.232  95.160   
         2       93.282  90.477  91.001  93.639  94.445  94.715  95.519   
         3       92.730  89.557  90.779  93.273  93.900  94.583  95.589   
         all     92.840  89.734  90.576  93.220  94.020  94.510  95.421   

alpha                                                    
weight_label    $w_{2}$ $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$  
clientes classe                                          
5        1        0.275   0.024   1.207   0.031   0.962  
         2        0.186   0.136   1.708   0.063   0.425  
         3        0.393   0.115   2.709   0.870   0.410  
         all      0.285   0.091   1.872   0.321   0.600  
10       1       41.296  55.744  68.142  40.826  67.773  
         2       41.391  42.646  67.361  38.206  65.434  
         3       43.022  47.122  69.204  40.249  69.984  
         all     41.903  48.504  68.236  39.760  67.731  
20       1       93.290  94.666  96.308  96.266  97.348  
         2       93.916  95.358  96.565  96.508  97.680  
         3       91.706  94.950  96.539  96.549  97.507  
         all     92.974  94.967  96.472  96.442  97.515

\begin{table}[t]
\centering
\caption{Average gap (\%) by clients/class (rows) and $\alpha$/weight (columns), without outliers.}
\label{tab:gap_alpha_weight}
\begin{tabular}{llcccccccccccc}
\toprule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{}} & \multicolumn{6}{c}{\textbf{$\alpha=0.01$}} & \multicolumn{6}{c}{\textbf{$\alpha=0.99$}} \\
\cmidrule(lr){3-8} \cmidrule(lr){9-14}
\multicolumn{1}{c}{\textbf{Clientes}} & \multicolumn{1}{c}{\textbf{Classe}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} \\
\midrule
\multirow{4}{*}{\textbf{5 clients}} & I & 4,0\% & 3,